In [1]:
import pandas as pd
import os
from pathlib import Path 

# **reusable functions**

In [2]:
def print_line(num):
    print('=' * num)
    
def print_txt(text, num=50):
    print_line(num)
    print(text)
    print_line(num)

In [3]:
def check_the_order(column):
    is_ordered_old_new = column.is_monotonic_increasing
    is_ordered_new_old = column.is_monotonic_decreasing
    return is_ordered_old_new, is_ordered_new_old

# **Import data**

In [4]:
# RAW_DATA_DIR = '/kaggle/input/datasets/karapy/azure-1/dataset-electric.csv'

RAW_DATA_DIR = Path('../data/raw/dataset-electric.csv')
SAVE_DATA_DIR = Path('../data/processed')

df = pd.read_csv(RAW_DATA_DIR)

# **Cleaning**

In [5]:


# df['Datetime'] = df['Datetime'].str.strip()
df['Datetime'] = pd.to_datetime(df['Datetime'])

In [6]:
print_txt('check if the Datetime column is ordered correctly')

is_ordered_old_new, is_ordered_new_old = check_the_order(df['Datetime'])

print(f'Ascending order  : {is_ordered_old_new}')
print(f'Descending order : {is_ordered_new_old}') 

check if the Datetime column is ordered correctly
Ascending order  : False
Descending order : False


In [7]:
print_txt('check the rows with wrong order ')

df.loc[df['Datetime'].diff() < pd.Timedelta(0)]

check the rows with wrong order 


,Datetime,PJME_MW
24,2002-12-30 01:00:00,27526.0
48,2002-12-29 01:00:00,27084.0
72,2002-12-28 01:00:00,29344.0
96,2002-12-27 01:00:00,28445.0
120,2002-12-26 01:00:00,26705.0
...,...,...
145246,2018-01-05 01:00:00,39667.0
145270,2018-01-04 01:00:00,34494.0
145294,2018-01-03 01:00:00,38011.0
145318,2018-01-02 01:00:00,37679.0


In [8]:
print_txt('sort the datetime column ascending')
df = df.sort_values(by='Datetime', ascending=True)

print('done')

sort the datetime column ascending
done


In [9]:
print_txt('Check if the order is fixed after sort_values')

is_ordered_old_new, _ = check_the_order(df['Datetime'])
print(f'Ascending order  : {is_ordered_old_new}')

Check if the order is fixed after sort_values
Ascending order  : True


In [10]:
SAVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
print_txt('Drop duplicated Datetime rows')
df.drop_duplicates(subset='Datetime',inplace=True)
print(f'New shape : {df.shape}')
print(f'columns   : {df.shape[0]}')
print(f'rows      : {df.shape[1]}')

Drop duplicated Datetime rows
New shape : (145392, 2)
columns   : 145392
rows      : 2


In [12]:
print_txt('Handle the missing Hours gap')

df = df.set_index('Datetime')
df = df.asfreq('h')

df['PJME_MW'] = df['PJME_MW'].interpolate(method='time', limit_area='inside')

df = df.reset_index()

Handle the missing Hours gap


In [13]:
df.to_csv(Path.joinpath(SAVE_DATA_DIR, 'cleaned_dataset-electric.csv'), index=False)